# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MRazaRashid/FlyRank_Week1/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:

import os, sys, subprocess, pandas as pd
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found.")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

Working dir: /content/flyrank-ml-internship-starter
Starter data found.


In [4]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
feature_cols = ["impressions_90d", "content_age_days", "days_since_last_update",
                 "avg_position", "ctr", "engagement_rate", "word_count"]

In [6]:
X = df[feature_cols].fillna(0)
y = (df["trend_direction"] == "down").astype(int)

X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_naive = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
model_naive.fit(X_train_naive, y_train_naive)
naive_auc = roc_auc_score(y_test_naive, model_naive.predict_proba(X_test_naive)[:, 1])

# HONEST split: grouped by client (same as Week 5) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

X_train_g = train_df[feature_cols].fillna(0)
y_train_g = (train_df["trend_direction"] == "down").astype(int)
X_test_g = test_df[feature_cols].fillna(0)
y_test_g = (test_df["trend_direction"] == "down").astype(int)

model_grouped = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
model_grouped.fit(X_train_g, y_train_g)
grouped_auc = roc_auc_score(y_test_g, model_grouped.predict_proba(X_test_g)[:, 1])

comparison = pd.DataFrame({
    "Split type": ["Naive random split", "Client-grouped split"],
    "ROC AUC": [naive_auc, grouped_auc]
})
print(comparison)

             Split type   ROC AUC
0    Naive random split  0.753483
1  Client-grouped split  0.586716


Under a naive random split, the model reports ROC AUC of 0.753, a strong result. However, under the honest client-grouped split (holding entire clients out of training), AUC drops to 0.587, much closer to random guessing. This ~0.17 point gap reveals that a meaningful share of the naive split's apparent performance came from the model partially recognizing client-specific patterns it had already seen in training, rather than learning signals that generalize to genuinely new clients. The grouped result (0.587) is the number I'm keeping as my honest, trustworthy estimate of how this model would likely perform on a client it has never encountered, which is the realistic use case for this tool in production. This also matches the same 0.587 result already reported in my Week 5 notebook, confirming that Week 5's split was already done correctly.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
leakage_checklist = {
    "Are any features calculated after the decision point?": "No — all features (impressions_90d, content_age_days, days_since_last_update, avg_position, ctr, engagement_rate, word_count) are current-state or trailing 90-day metrics available before any decision.",
    "Does the feature window overlap the target window?": "No — trend_direction (the label source) is a separately-computed field, not built from summing the same feature window.",
    "Did any product output (health_score, action_type) slip in as a feature?": "No — these fields are not present in the starter dataset at all.",
    "Does a derived field secretly encode the target?": "Caught and fixed in Week 5: an early baseline version included trend_direction directly in the score, inflating precision@50 to a false 1.00. This was identified and removed.",
    "Are duplicate/related rows split across train and test unfairly?": "Addressed via client-grouped splitting — no client appears in both train and test.",
    "Are you testing on clients the model hasn't effectively already seen?": "Yes, confirmed — the grouped split explicitly holds out entire clients.",
}

for q, a in leakage_checklist.items():
    print(f"Q: {q}\nA: {a}\n")

Q: Are any features calculated after the decision point?
A: No — all features (impressions_90d, content_age_days, days_since_last_update, avg_position, ctr, engagement_rate, word_count) are current-state or trailing 90-day metrics available before any decision.

Q: Does the feature window overlap the target window?
A: No — trend_direction (the label source) is a separately-computed field, not built from summing the same feature window.

Q: Did any product output (health_score, action_type) slip in as a feature?
A: No — these fields are not present in the starter dataset at all.

Q: Does a derived field secretly encode the target?
A: Caught and fixed in Week 5: an early baseline version included trend_direction directly in the score, inflating precision@50 to a false 1.00. This was identified and removed.

Q: Are duplicate/related rows split across train and test unfairly?
A: Addressed via client-grouped splitting — no client appears in both train and test.

Q: Are you testing on client

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest original claim**: "Random Forest beats the baseline rule at prioritizing pages for review, achieving more than double the precision."

**Rewritten in safe language:**
"On this dataset, under a client-grouped test split, the Random Forest model achieved an observed precision@50 of 0.66, compared to 0.30 for the rule-based baseline. This result is decision-support only: it suggests the model may help a reviewer prioritize more effectively than the existing rule, but it does not establish that acting on these recommendations would improve outcomes, since no causal experiment was run. The result also reflects one dataset, one label definition (trend_direction == 'down', itself a same-window proxy rather than a true future outcome), and one train/test split — it should be read as an initial, promising signal, not a guaranteed or general result."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.